# Neural-Agent Training on Kaggle (v2)

Train the proprietary NeuralAgent model on Kaggle's free GPU (T4, 15.6GB VRAM).

**Neural-Agent is PRIVATE and PAID - not published on PyPI.**

v2 uses the bundled `train_kaggle.py` script (single command) instead of inline cells.

## Setup
1. Enable GPU: **Settings > Accelerator > GPU T4 x2**
2. Enable Internet: **Settings > Internet > On**
3. Run all cells

## Step 0: Upload Neural-Agent (one-time setup)

Neural-Agent is private. Upload it as a Kaggle dataset:
1. Go to kaggle.com -> Datasets -> New Dataset
2. Upload `neural-agent-kaggle.zip` (build with `python build_kaggle_zip.py` in Neural-Agent repo)
3. Set dataset to **Private**
4. Copy dataset slug (e.g. `yourname/neural-agent-private`)
5. Update `DATASET_SLUG` below if slug differs from `yourname/neural-agent-private`

In [ ]:
DATASET_SLUG = "yourname/neural-agent-private"  # <-- EDIT THIS
print(f"Using dataset: {DATASET_SLUG}")

## Step 1: Install Dependencies

In [ ]:
!pip install neuraldbg transformers peft trl bitsandbytes datasets accelerate -q
!pip install /kaggle/input/{DATASET_SLUG.split('/')[-1]}/neural-agent-kaggle.zip -q

## Step 2: Train (single command)

Runs `train_kaggle.py` from the uploaded dataset:
- Formats triplets -> train.jsonl + val.jsonl
- Trains Qwen2-0.5B with QLoRA (4-bit + LoRA)
- Merges LoRA into base model
- Saves to `/kaggle/working/neuralagent_output/`

In [ ]:
%run /kaggle/input/{DATASET_SLUG.split('/')[-1]}/train_kaggle.py

## Step 3: Verify Output

In [ ]:
from pathlib import Path
import os

out = Path("/kaggle/working/neuralagent_output")
for f in sorted(out.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.relative_to(out)}  ({size_mb:.2f} MB)")

## Step 4: Download

Click **Save Version** -> **Output** tab -> download `merged/` directory.

Or save the LoRA adapter only (smaller):
- `checkpoints/final/adapter_model.safetensors` (LoRA only)

Then load in Neural-Agent locally with `peft.PeftModel.from_pretrained(...)`.